# Spline & Systematic Shift Checks — WC Reconstructed $E_\nu$

Plot the Wire-Cell reconstructed neutrino energy for all (generic-preselected) MC events,
overlay **every spline-morph shift** and **every multisim systematic shift**, and flag any
that are more extreme than expected.

Motivation: in external code (PROfit, which reads `minimal_withspline_df.root`) a few bins
showed unexpectedly large uncertainties. This notebook reproduces the per-knob / per-universe
spectra directly from the intermediate parquets so we can see *which* knob, *which* bin, and
*why*.

Two weight families are checked separately:

* **Spline shifts** — per-knob multi-sigma morph splines in `spline_weights_df.parquet`
  (e.g. `MaCCQE_UBGenie` = 7 sigma points; the center point is `TunedCentralValue`).
  GENIE `*_UBGenie` knobs are *absolute* GENIE weights (they already include the tune), so
  they replace `wc_weight_cv`; flux/reint knobs are *relative* multipliers on the full weight.
* **Systematic (multisim) shifts** — universe throws in `presel_weights_df.parquet`
  (`All_UBGenie` = 500, `flux_all` = 1000, `reint_all` = 1000), plus `weightsReint`.

Key subtlety we test for: `src/systematics.py::create_universe_histograms` **caps** weights
(`>30 → 1`, `<0 → 1`, `nan/inf → 1`) before building covariances. Code that does *not* cap
(e.g. a raw PROfit read) will show much larger bins wherever a single event has a pathological
weight. Every plot below shows **uncapped** vs **capped** so the difference is explicit.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 110

from src.file_locations import intermediate_files_location as IL

# ----------------------------------------------------------------------------- config
# Default = run 4b only, because minimal_withspline_df.root (the external PROfit input,
# where the too-large uncertainties were seen) is built from run 4b with the
# run4b_only_wc_net_weight POT normalization. This is also fast and memory-safe.
#
# To check ALL runs instead, set:
#   SELECTED_RUN_PERIODS = ["1","2","3","4a","4b","4c","4d","5"]
#   NET_WEIGHT_VAR       = "wc_net_weight_full_pred"
# (the big multisim section below then needs ~1000 x N_events floats in RAM; restrict runs
#  if that is too much.)
SELECTED_RUN_PERIODS = ["4b"]
NET_WEIGHT_VAR       = "run4b_only_wc_net_weight"

VAR  = "wc_kine_reco_Enu"
BINS = np.linspace(0, 2000, 21)          # MeV, matches the external code binning
BIN_CENTERS = 0.5 * (BINS[:-1] + BINS[1:])
NBINS = len(BINS) - 1

CAP = 30.0   # weight cap used in src/systematics.py; weights >CAP, <0, or non-finite -> 1

print(f"run periods : {SELECTED_RUN_PERIODS}")
print(f"net weight  : {NET_WEIGHT_VAR}")
print(f"var / bins  : {VAR}, {NBINS} bins over [{BINS[0]}, {BINS[-1]}] MeV")

## 1. Load the nominal MC spectrum

We take the event set from `spline_weights_df.parquet` (already preselected to
`wc_kine_reco_Enu > 0`, MC overlays only), and attach the reco energy and the nominal
net weight from `all_df.parquet`. Data / EXT have no systematic weights and are excluded by
the inner join.

In [ ]:
# small spline-morph knob columns (exclude the 1000-wide weightsReint; keep <=50 universes)
spline_scan = pl.scan_parquet(f"{IL}/spline_weights_df.parquet")
spline_schema = spline_scan.collect_schema()
_probe = spline_scan.select(
    [pl.col(c).list.len().alias(c) for c, t in spline_schema.items() if isinstance(t, pl.List)]
).head(1).collect()
KNOB_NUNIV = {c: int(_probe[c][0]) for c in _probe.columns}
SPLINE_KNOBS = sorted([c for c, n in KNOB_NUNIV.items() if 1 < n <= 50 and c != "weightsReint"])
print(f"{len(SPLINE_KNOBS)} spline-morph knobs (universe counts: "
      f"{sorted(set(KNOB_NUNIV[c] for c in SPLINE_KNOBS))})")

# RSE alone is NOT unique across filetypes (data/ext/overlays reuse run/subrun/event), so
# every join uses filename+run+subrun+event -- the same key create_rw_syst_df.py writes with.
JOIN_KEYS = ["filename", "run", "subrun", "event"]

# event set for the selected run periods, plus all small knob columns
spline_sel = (
    spline_scan
    .filter(pl.col("detailed_run_period").is_in(SELECTED_RUN_PERIODS))
    .select(JOIN_KEYS + SPLINE_KNOBS)
)

# nominal reco energy + weights from all_df (a few % of preselected events are dropped by
# all_df postprocessing; the inner join keeps only events with full nominal info)
nominal_scan = (
    pl.scan_parquet(f"{IL}/all_df.parquet")
    .filter(pl.col("detailed_run_period").is_in(SELECTED_RUN_PERIODS))
    .select(JOIN_KEYS + ["filetype", VAR, NET_WEIGHT_VAR, "wc_weight_cv"])
)

df = (
    spline_sel.join(nominal_scan, on=JOIN_KEYS, how="inner")
    .collect(engine="streaming")
)
print(f"loaded {df.height:,} MC events")
print(df.select("filetype").to_series().value_counts().sort("count", descending=True))

In [ ]:
vals = df[VAR].to_numpy().astype(np.float64)
net_w = df[NET_WEIGHT_VAR].to_numpy().astype(np.float64)
cv_w  = df["wc_weight_cv"].to_numpy().astype(np.float64)

# GENIE *_UBGenie knobs are absolute (include the tune) -> replace wc_weight_cv:
#   event weight = net_weight / wc_weight_cv, then multiply by the knob weight.
# flux/reint knobs are relative multipliers on the full net_weight.
with np.errstate(divide="ignore", invalid="ignore"):
    non_genie_cv = np.where(cv_w != 0, net_w / cv_w, 0.0)
non_genie_cv = np.nan_to_num(non_genie_cv, nan=0.0, posinf=0.0, neginf=0.0)

# precompute bin index once
bin_idx = np.searchsorted(BINS, vals, side="right") - 1
in_range = (bin_idx >= 0) & (bin_idx < NBINS)

def _hist(weights):
    w = np.where(in_range, weights, 0.0)
    return np.bincount(bin_idx[in_range], weights=w[in_range], minlength=NBINS)

nominal_hist = _hist(net_w)
print("nominal counts per bin:")
for c, n in zip(BIN_CENTERS, nominal_hist):
    print(f"  {c:7.0f} MeV : {n:10.2f}")
print(f"total = {nominal_hist.sum():.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(BIN_CENTERS, weights=nominal_hist, bins=BINS, histtype="stepfilled",
        color="tab:blue", alpha=0.35, edgecolor="tab:blue")
ax.set_xlabel(r"WC reconstructed $E_\nu$ (MeV)")
ax.set_ylabel(f"Weighted events ({NET_WEIGHT_VAR})")
ax.set_title(f"Nominal MC spectrum — runs {'+'.join(SELECTED_RUN_PERIODS)}  ({df.height:,} events)")
ax.set_xlim(BINS[0], BINS[-1])
plt.tight_layout(); plt.show()

## 2. Spline-morph shifts (per knob)

For each knob we histogram every sigma-point universe and compare to the nominal spectrum.
We do this **uncapped** (raw weights, as external code would use) and **capped** (weights
`>30`, `<0`, or non-finite set to 1, matching `src/systematics.py`). A knob whose uncapped
band is far larger than its capped band has a pathological weight — the likely source of an
over-large bin.

`classify_knob` picks the event-weight base per the convention above.

In [ ]:
def classify_knob(name):
    if "Flux" in name or "Hadron" in name:
        return "flux"
    if "Reint" in name or "reint" in name:
        return "reint"
    return "genie"   # *_UBGenie, xsr_scc_*_SCC, etc.

def knob_universe_hists(name, cap=False):
    '''Return (nuniv, NBINS) array of per-universe histograms for a knob.

    Non-finite (nan/inf) weights are always neutralized to 1 (they are rare and would
    otherwise poison a whole bin). "capped" additionally applies the src/systematics.py
    caps (>30 -> 1, <0 -> 1), so uncapped-vs-capped isolates exactly that capping.'''
    n = KNOB_NUNIV[name]
    w2d = df[name].list.to_array(n).to_numpy().astype(np.float64)   # (E, n)
    w2d = np.nan_to_num(w2d, nan=1.0, posinf=1.0, neginf=1.0)
    if cap:
        w2d = np.where(w2d > CAP, 1.0, w2d)
        w2d = np.where(w2d < 0.0, 1.0, w2d)
    base = non_genie_cv if classify_knob(name) == "genie" else net_w
    hists = np.empty((n, NBINS))
    for u in range(n):
        hists[u] = _hist(base * w2d[:, u])
    return hists

# quick per-knob raw-weight extremes (independent of binning / convention)
records = []
for name in SPLINE_KNOBS:
    n = KNOB_NUNIV[name]
    w = df[name].list.to_array(n).to_numpy().astype(np.float64)
    finite = np.isfinite(w)
    records.append(dict(
        knob=name, nuniv=n, kind=classify_knob(name),
        wmax=np.nanmax(np.where(finite, w, np.nan)),
        wmin=np.nanmin(np.where(finite, w, np.nan)),
        n_gt_cap=int(np.sum(finite & (w > CAP))),
        n_neg=int(np.sum(finite & (w < 0))),
        n_nonfinite=int(np.sum(~finite)),
    ))
extreme_df = pl.DataFrame(records).sort("wmax", descending=True)
print("Per-knob raw spline-weight extremes (sorted by max weight):")
with pl.Config(tbl_rows=100, fmt_str_lengths=60):
    print(extreme_df)

In [ ]:
# per-knob band: max over bins of |universe/nominal - 1|, uncapped vs capped
safe_nom = np.where(nominal_hist > 0, nominal_hist, np.nan)
band_rows = []
uncapped_hists = {}
for name in SPLINE_KNOBS:
    hu = knob_universe_hists(name, cap=False)
    hc = knob_universe_hists(name, cap=True)
    uncapped_hists[name] = hu
    frac_u = np.abs(hu / safe_nom - 1.0)
    frac_c = np.abs(hc / safe_nom - 1.0)
    per_bin_u = np.nanmax(np.where(np.isfinite(frac_u), frac_u, np.nan), axis=0)
    per_bin_u = np.nan_to_num(per_bin_u, nan=-1.0)   # zero-nominal bins -> ignore in argmax
    band_rows.append(dict(
        knob=name, kind=classify_knob(name),
        max_frac_uncapped=float(np.nanmax(frac_u)),
        max_frac_capped=float(np.nanmax(frac_c)),
        worst_bin=int(np.argmax(per_bin_u)),
    ))
band_df = pl.DataFrame(band_rows).with_columns(
    (pl.col("max_frac_uncapped") / pl.col("max_frac_capped").clip(1e-9)).alias("uncapped_over_capped")
).sort("max_frac_uncapped", descending=True)
print("Per-knob worst single-bin fractional shift (uncapped vs capped):")
with pl.Config(tbl_rows=100, fmt_str_lengths=60):
    print(band_df)

In [ ]:
# bar chart: worst per-knob single-bin fractional shift, uncapped vs capped
bd = band_df.sort("max_frac_uncapped", descending=True)
names = bd["knob"].to_list()
y = np.arange(len(names))
fig, ax = plt.subplots(figsize=(9, max(4, 0.22 * len(names))))
ax.barh(y - 0.2, bd["max_frac_uncapped"].to_numpy(), height=0.4, color="tab:red", label="uncapped")
ax.barh(y + 0.2, bd["max_frac_capped"].to_numpy(), height=0.4, color="tab:blue", label="capped (>30,<0,inf -> 1)")
ax.set_yticks(y); ax.set_yticklabels(names, fontsize=7)
ax.invert_yaxis()
ax.set_xlabel("max over bins of |universe / nominal - 1|")
ax.set_title("Worst single-bin spline shift per knob")
ax.axvline(1.0, color="k", ls=":", lw=1)
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# overlay grid: every knob's universes as a ratio to nominal (uncapped)
ncols = 6
nrows = int(np.ceil(len(SPLINE_KNOBS) / ncols))
fig, axs = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 2.2 * nrows), squeeze=False)
axs = axs.flatten()
order = band_df["knob"].to_list()   # worst first
for k, name in enumerate(order):
    ax = axs[k]
    hu = uncapped_hists[name]
    n = hu.shape[0]
    colors = plt.cm.coolwarm(np.linspace(0, 1, n))
    for u in range(n):
        ax.step(BIN_CENTERS, hu[u] / safe_nom, where="mid", color=colors[u], lw=0.8)
    ax.axhline(1.0, color="k", lw=0.7)
    worst = band_df.filter(pl.col("knob") == name)["max_frac_uncapped"][0]
    title_color = "red" if worst > 1.0 else "black"
    ax.set_title(f"{name}\n(max dev {worst*100:.0f}%)", fontsize=6.5, color=title_color)
    ax.tick_params(labelsize=5)
    ax.set_xlim(BINS[0], BINS[-1])
for k in range(len(order), len(axs)):
    axs[k].remove()
fig.suptitle("Spline-morph universes / nominal  (uncapped, red title = >100% in some bin)", y=1.001)
plt.tight_layout(); plt.show()

## 3. Multisim systematic shifts

The canonical universe throws live in `presel_weights_df.parquet`:
`All_UBGenie` (500 universes, GENIE — replaces `wc_weight_cv`), `flux_all` (1000, relative)
and `reint_all` (1000, relative). We build the per-bin fractional 1σ band as
`std(universe_hists) / nominal`, again **uncapped** vs **capped**.

For all-runs this pulls ~1000 × N_events floats into RAM per systematic; it is fine for
run 4b. Universes are processed in chunks to bound memory.

In [ ]:
MULTISIMS = [("All_UBGenie", "genie"), ("flux_all", "flux"), ("reint_all", "reint")]

# join the multisim columns for our exact event set (same filename+RSE key)
keys = df.select(JOIN_KEYS)
multi = (
    keys.lazy().join(
        pl.scan_parquet(f"{IL}/presel_weights_df.parquet")
        .filter(pl.col("detailed_run_period").is_in(SELECTED_RUN_PERIODS))
        .select(JOIN_KEYS + [m for m, _ in MULTISIMS]),
        on=JOIN_KEYS, how="left",
    ).collect(engine="streaming")
)
# align to df row order
multi = df.select(JOIN_KEYS).join(multi, on=JOIN_KEYS, how="left")
null_counts = {m: multi[m].null_count() for m, _ in MULTISIMS}
print(f"multisim rows: {multi.height:,} (df rows: {df.height:,})")
print(f"null (unmatched) rows per systematic: {null_counts}")
assert all(v == 0 for v in null_counts.values()), \
    "some events have no multisim weights; they would poison the band -- investigate the join"
# materialize each multisim as a (E, nuniv) float64 array (run 4b: ~3.6 GB each, freed after use)
multi_arr = {m: multi[m].list.to_array(multi[m].list.len().max()).to_numpy().astype(np.float64)
             for m, _ in MULTISIMS}

In [ ]:
def multisim_band(name, kind, cap):
    base = non_genie_cv if kind == "genie" else net_w
    w2d = np.nan_to_num(multi_arr[name], nan=1.0, posinf=1.0, neginf=1.0)
    if cap:
        w2d = np.where(w2d > CAP, 1.0, w2d)
        w2d = np.where(w2d < 0.0, 1.0, w2d)
    n = w2d.shape[1]
    hists = np.empty((n, NBINS))
    for u in range(n):
        hists[u] = _hist(base * w2d[:, u])
    frac_sigma = hists.std(axis=0) / safe_nom
    return np.nan_to_num(frac_sigma, nan=0.0)

multi_frac = {}
for name, kind in MULTISIMS:
    multi_frac[(name, "uncapped")] = multisim_band(name, kind, cap=False)
    multi_frac[(name, "capped")]   = multisim_band(name, kind, cap=True)
    print(f"{name}: done")

In [ ]:
fig, axs = plt.subplots(1, len(MULTISIMS), figsize=(5 * len(MULTISIMS), 4), squeeze=False)
axs = axs[0]
for ax, (name, kind) in zip(axs, MULTISIMS):
    ax.step(BIN_CENTERS, 100 * multi_frac[(name, "uncapped")], where="mid",
            color="tab:red", label="uncapped")
    ax.step(BIN_CENTERS, 100 * multi_frac[(name, "capped")], where="mid",
            color="tab:blue", label="capped")
    ax.set_title(name); ax.set_xlabel(r"WC reco $E_\nu$ (MeV)")
    ax.set_ylabel(r"fractional 1$\sigma$ (%)")
    ax.set_xlim(BINS[0], BINS[-1]); ax.legend()
fig.suptitle("Multisim per-bin fractional uncertainty")
plt.tight_layout(); plt.show()

## 4. Combined per-bin uncertainty and summary

Sum every source (all spline knobs + all multisims) in quadrature per bin, uncapped vs
capped, to see which bins blow up when weights are not capped — the direct analogue of the
external-code symptom.

In [ ]:
# spline knobs: per-bin fractional 1-sigma as spread across each knob's universes.
# Uncapped values can be astronomically large (pathological knobs like kminus), so we sum
# in quadrature with overflow ignored and treat any non-finite result as "off scale".
def knob_frac_sigma(name, cap):
    h = knob_universe_hists(name, cap=cap)
    return h.std(axis=0) / safe_nom   # may be inf where safe_nom is nan or weights huge

with np.errstate(over="ignore", invalid="ignore"):
    tot_u = np.zeros(NBINS); tot_c = np.zeros(NBINS)
    for name in SPLINE_KNOBS:
        tot_u += np.nan_to_num(knob_frac_sigma(name, cap=False), nan=0.0) ** 2
        tot_c += np.nan_to_num(knob_frac_sigma(name, cap=True), nan=0.0) ** 2
    for name, _ in MULTISIMS:
        tot_u += np.nan_to_num(multi_frac[(name, "uncapped")], nan=0.0) ** 2
        tot_c += np.nan_to_num(multi_frac[(name, "capped")], nan=0.0) ** 2
    tot_u = np.sqrt(tot_u); tot_c = np.sqrt(tot_c)

# bins whose uncapped total is non-finite or absurd -> off scale; plot capped, mark them
off_scale = ~np.isfinite(tot_u) | (tot_u > 10 * np.nanmax(tot_c))
tot_u_plot = np.where(off_scale, np.nan, tot_u)
ymax = 100 * np.nanmax(np.concatenate([tot_c, np.nan_to_num(tot_u_plot)])) * 1.3

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.step(BIN_CENTERS, 100 * tot_u_plot, where="mid", color="tab:red", lw=2, label="uncapped total")
ax.step(BIN_CENTERS, 100 * tot_c, where="mid", color="tab:blue", lw=2, label="capped total")
if off_scale.any():
    ax.scatter(BIN_CENTERS[off_scale], np.full(off_scale.sum(), ymax * 0.95),
               marker="^", color="tab:red", s=60, zorder=5,
               label="uncapped OFF SCALE (pathological knob)")
ax.set_ylim(0, ymax)
ax.set_xlabel(r"WC reconstructed $E_\nu$ (MeV)")
ax.set_ylabel(r"total fractional 1$\sigma$ (%)")
ax.set_title("Total systematic fractional uncertainty per bin (splines + multisims, quadrature)")
ax.set_xlim(BINS[0], BINS[-1]); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("bin           nominal    capped%     uncapped%   ratio")
for i in range(NBINS):
    r = tot_u[i] / tot_c[i] if tot_c[i] > 0 else np.nan
    flag = "  <-- OFF SCALE" if off_scale[i] else ("  <-- blows up" if r > 2 else "")
    u_str = "    off-scale" if off_scale[i] else f"{100*tot_u[i]:10.1f}"
    print(f"{BINS[i]:5.0f}-{BINS[i+1]:<5.0f} {nominal_hist[i]:11.2f} "
          f"{100*tot_c[i]:8.1f} {u_str}   {r:8.2g}{flag}")

In [ ]:
# Which knob/bin dominates the uncapped total? Rank (knob, bin) by uncapped fractional sigma.
def _worst(fs):
    fs = np.where(np.isfinite(fs), fs, np.inf)   # inf -> off-scale, still ranked to top
    b = int(np.argmax(fs))
    return b, float(fs[b])

rows = []
with np.errstate(over="ignore", invalid="ignore"):
    for name in SPLINE_KNOBS:
        b, v = _worst(knob_frac_sigma(name, cap=False))
        rows.append(dict(source=name, kind=classify_knob(name), bin=b,
                         bin_range=f"{BINS[b]:.0f}-{BINS[b+1]:.0f}", frac_sigma_pct=100 * v))
    for name, kind in MULTISIMS:
        b, v = _worst(multi_frac[(name, "uncapped")])
        rows.append(dict(source=name, kind=kind, bin=b,
                         bin_range=f"{BINS[b]:.0f}-{BINS[b+1]:.0f}", frac_sigma_pct=100 * v))
rank = pl.DataFrame(rows).sort("frac_sigma_pct", descending=True, nulls_last=True)
print("Top contributors to the UNCAPPED per-bin uncertainty (inf = pathological/off-scale):")
with pl.Config(tbl_rows=30, fmt_str_lengths=60):
    print(rank.head(30))

## 5. Kaon PrimaryHadron knob pathologies (kminus, kplus, kzero)

All three kaon multisigma knobs are broken at eventweight production. Root causes confirmed in
`ubsim/ubsim/EventWeight/Calculators/BNBPrimaryHadron/`: the multisigma mode was bolted onto
machinery designed for multisim (per-universe vectors of Gaussian draws, one per fit parameter,
combined via Cholesky in `WeightCalc::MultiGaussianSmearing`), and each calculator breaks at the
seam it touches:

| knob | bug | symptom |
|---|---|---|
| `kminus` (Normalization) | 1D `fmultisigmas[weight_index]` OOB read (array sized `2*fNmultisims`=14, only 7 sigmas) **+** `w=1+σ>0` rejection loop backfills from the OOB garbage | stored list is `[1,2,3,4, garbage+1, ...]` — sigma alignment destroyed, garbage up to 1.7e243 |
| `kplus` (FeynmanScaling) | `fWeightArray[i][j] = fmultisigmas[j]` — **parameter** index `j` instead of **universe** index `i`, so every universe gets the identical throw `[-3,-2,-1,0,1,2,3]` (FS has exactly 7 params → no OOB) | all 7 universes bit-identical; plausible-looking values 0.65–1.0 with zero uncertainty content |
| `kzero` (SanfordWang) | same `[j]` typo **+** OOB (SW has 9 params > 7 sigmas); the fixed smeared params fail an event-independent validity check (`c1<0 ‖ c3<0 ‖ c6<0`, SanfordWangWeightCalc.cxx:230 pads) | every universe of every K⁰-parent event = **−9898 sentinel** |

Dropping the σ=−3,−2 universes does **not** fix any of these. The kminus fix is analytic
(normalization knob → `w(σ) = max(1+σ, 0)` for `[1,2,3,4]`-prefix events); kplus/kzero are
unrecoverable from the stored data. The 1000-universe `flux_all` multisim uses the calculators in
their designed mode and is unaffected. These checks run over **all run periods** (the knob columns
are cheap to load).

In [ ]:
# load just the kaon knob columns for ALL runs
KAON_COLS = ["kminus_PrimaryHadronNormalization", "kplus_PrimaryHadronFeynmanScaling",
             "kzero_PrimaryHadronSanfordWang"]
kaon_df = (
    pl.scan_parquet(f"{IL}/spline_weights_df.parquet")
    .select(["filename", "detailed_run_period", "filetype"] + KAON_COLS)
    .collect(engine="streaming")
)
print(f"loaded {kaon_df.height:,} events (all runs)")

In [ ]:
# ---- kminus: shifted + garbage ----
km = kaon_df["kminus_PrimaryHadronNormalization"].list.to_array(7).to_numpy().astype(np.float64)
km = np.nan_to_num(km, nan=1.0, posinf=np.inf, neginf=-np.inf)

# K- parent events carry the [1,2,3,4] prefix = w(sigma=0..+3) shifted into the slots meant
# for sigma=-3..0 (the sigma<=-1 universes were rejected by the w=1+sigma>0 check)
kminus_parent = (km[:, 0] == 1) & (km[:, 1] == 2) & (km[:, 2] == 3) & (km[:, 3] == 4)
huge = (km > CAP).any(axis=1)
print(f"K- parent events (prefix [1,2,3,4]) : {kminus_parent.sum():,} / {len(km):,}")
print(f"  with garbage > {CAP:.0f}              : {(kminus_parent & huge).sum():,}")
print(f"  huge events NOT matching prefix   : {(huge & ~kminus_parent).sum():,}  (expect 0)")
rest = km[~kminus_parent]
print(f"  non-K- rows exactly all-1         : {(rest == 1).all(axis=1).sum():,} / {len(rest):,}  (expect all)")

print("\ndistinct tail patterns (slots 4,5,6 = garbage+1; distinct patterns = distinct heap layouts):")
tails = {}
for i in np.where(kminus_parent)[0]:
    tails[tuple(km[i, 4:7])] = tails.get(tuple(km[i, 4:7]), 0) + 1
for k, c in sorted(tails.items(), key=lambda x: -x[1]):
    print(f"  count={c:5d}: [{k[0]:.6e}, {k[1]:.6e}, {k[2]:.6e}]")

print("\nK- parent events by run period / filetype:")
print(kaon_df.filter(pl.Series(kminus_parent))
      .group_by(["detailed_run_period", "filetype"]).len()
      .sort(["detailed_run_period", "filetype"]))

print("\ncorrect analytic reconstruction for these events, sigma grid [-3..+3]:")
print("  w(sigma) = max(1 + sigma, 0)  ->  [0, 0, 0, 1, 2, 3, 4]")

In [ ]:
# ---- kplus: all universes identical (no uncertainty content) ----
kp = kaon_df["kplus_PrimaryHadronFeynmanScaling"].list.to_array(7).to_numpy().astype(np.float64)
kp_parent = (kp != 1).any(axis=1)
kp_par = kp[kp_parent]
all_same = (kp_par == kp_par[:, [0]]).all(axis=1)
print(f"K+ parent events (any weight != 1)      : {kp_parent.sum():,}")
print(f"  all 7 universes bit-identical         : {all_same.sum():,} / {len(kp_par):,}  (expect all = broken)")
print(f"  stored value range                    : {kp_par.min():.4f} .. {kp_par.max():.4f}")
print(f"  note w('sigma=0') != 1: every universe evaluates the same wrongly-shifted parameter")
print(f"  vector [-3,-2,-1,0,1,2,3] applied across the 7 FS fit parameters")

# ---- kzero: all-sentinel ----
kz = kaon_df["kzero_PrimaryHadronSanfordWang"].list.to_array(7).to_numpy().astype(np.float64)
kz_parent = (kz != 1).any(axis=1)
kz_par = kz[kz_parent]
all_sent = (kz_par == -9898.0).all(axis=1)
print(f"\nK0 parent events (any weight != 1)      : {kz_parent.sum():,}")
print(f"  all 7 universes == -9898 sentinel     : {all_sent.sum():,} / {len(kz_par):,}  (expect all = broken)")
print(f"  (sentinel written by the 'calculator returned false too often' padding loop,")
print(f"   PrimaryHadronSanfordWangWeightCalc.cxx:230)")

## 6. Dead knob: ThetaDelta2NRad_UBGenie

`ThetaDelta2NRad_UBGenie` (the Δ→Nγ radiative-decay angular-distribution knob — directly
signal-relevant for a single-photon analysis) **never varies**: both universes equal the tuned
central value exactly, in every file and every run period, in both the spline tree and the
multisim `presel_weights_df` tree. It is dead at eventweight production, not in our processing —
and it silently contributes exactly zero to the unisim covariance in
`src/systematics.py::create_rw_frac_cov_matrices`.

The airtight test: select **true Δ-radiative events** (`wc_truth_NCDeltaRad`, which flags both NC
and CC Δ radiative) and compare against `RDecBR1gamma_UBGenie` (radiative branching-ratio knob).
RDecBR1gamma varies with textbook values for essentially all of them — proving the
radiative-decay reweighting machinery fires for these exact events — while ThetaDelta2NRad stays
frozen at the tune.

In [ ]:
# per-file check over ALL runs: does ThetaDelta2NRad ever vary / ever differ from the tune?
td_df = (
    pl.scan_parquet(f"{IL}/spline_weights_df.parquet")
    .select(["filename", "run", "subrun", "event",
             "ThetaDelta2NRad_UBGenie", "RDecBR1gamma_UBGenie", "TunedCentralValue_UBGenie"])
    .collect(engine="streaming")
)
td = td_df["ThetaDelta2NRad_UBGenie"].list.to_array(2).to_numpy().astype(np.float64)
rd = td_df["RDecBR1gamma_UBGenie"].list.to_array(7).to_numpy().astype(np.float64)
tune = td_df["TunedCentralValue_UBGenie"].list.to_array(1).to_numpy().astype(np.float64)[:, 0]

td_varies = td[:, 0] != td[:, 1]
finite = np.isfinite(td).all(axis=1) & np.isfinite(tune)
eq_tune = np.isclose(td[:, 0], tune, rtol=1e-5) & np.isclose(td[:, 1], tune, rtol=1e-5)
print(f"ThetaDelta2NRad varies (univ0 != univ1) : {td_varies.sum():,} / {len(td):,}")
print(f"both universes == tune (finite events)  : {(eq_tune & finite).sum():,} / {finite.sum():,}")
print(f"RDecBR1gamma varies (machinery works)   : {(rd != rd[:, [0]]).any(axis=1).sum():,}")

per_file = (td_df.with_columns(pl.Series("td_varies", td_varies))
            .group_by("filename").agg(pl.len().alias("n_events"), pl.col("td_varies").sum())
            .sort("filename"))
with pl.Config(tbl_rows=40, fmt_str_lengths=100):
    print(per_file)

In [ ]:
# airtight: true Delta-radiative events specifically
truth = (
    pl.scan_parquet(f"{IL}/all_df.parquet")
    .filter(pl.col("wc_truth_NCDeltaRad") == True)
    .select(["filename", "run", "subrun", "event"])
    .collect(engine="streaming")
)
j = truth.join(td_df, on=["filename", "run", "subrun", "event"], how="inner")
td_j = j["ThetaDelta2NRad_UBGenie"].list.to_array(2).to_numpy().astype(np.float64)
rd_j = j["RDecBR1gamma_UBGenie"].list.to_array(7).to_numpy().astype(np.float64)
tune_j = j["TunedCentralValue_UBGenie"].list.to_array(1).to_numpy().astype(np.float64)[:, 0]

print(f"true Delta-radiative events (all_df, NC+CC): {truth.height:,}; matched in spline tree: {j.height:,}")
print(f"  ThetaDelta2NRad varies    : {(td_j[:, 0] != td_j[:, 1]).sum()} / {len(td_j)}")
print(f"  ThetaDelta2NRad == tune   : {(np.isclose(td_j[:, 0], tune_j) & np.isclose(td_j[:, 1], tune_j)).sum()} / {len(td_j)}")
print(f"  RDecBR1gamma varies       : {(rd_j != rd_j[:, [0]]).any(axis=1).sum()} / {len(rd_j)}")
i = np.where((rd_j != rd_j[:, [0]]).any(axis=1))[0][0]
print(f"\nexample true Delta-radiative event:")
print(f"  RDecBR1gamma    : {[round(x, 4) for x in rd_j[i]]}   <- knob machinery fires")
print(f"  ThetaDelta2NRad : {list(td_j[i])}  (tune = {tune_j[i]:.4f})   <- frozen at tune")

## How to read this

* A knob with **`max_frac_uncapped` >> `max_frac_capped`** (large `uncapped_over_capped`) has a
  pathological weight in at least one event — that single event dominates a bin only when the
  weights are *not* capped. This is the mechanism behind "a few bins with way too large
  uncertainties" in external code that reads the raw weights.
* The **raw-weight extremes** table pinpoints it directly: look at `wmax`, `n_gt_cap`, `n_neg`,
  and `n_nonfinite`. `src/systematics.py` sets any weight `>30`, `<0`, or non-finite to 1, so
  the covariance-based bands there are immune; PROfit / raw reads are not.
* If a bin is flagged **OFF SCALE** in the total table, check the ranking table for the source
  and go back to the offending knob to decide whether to cap it, drop the event, or fix the
  upstream spline.
* Capping is **not sufficient** for the kaon knobs (section 5): kminus is *misaligned* (slot k is
  not sigma k), and kplus/kzero carry no uncertainty information at all. Reconstruct kminus
  analytically; kplus/kzero need regenerated weights upstream.
* A knob can also fail **silently small**: a flat response (section 6, ThetaDelta2NRad; also
  kplus) fits a spline of zero uncertainty and simply vanishes from the error budget. Look for
  panels in the section-2 grid whose universes all overlap at ~1.
* To scan all runs in sections 1-4, edit `SELECTED_RUN_PERIODS`/`NET_WEIGHT_VAR` in the config
  cell (mind the multisim RAM note); sections 5-6 always run over all runs.